# text2concept-pytorch — Synthetic Sanity Check
Install → import → forward → backward → shapes

In [ ]:
!pip install text2concept-pytorch -q

In [ ]:
import torch
import torch.nn as nn
from text2concept_pytorch import Text2Concept, Text2ConceptAligner, Trainer, IMAGENET_TEMPLATES
print('imports ok')

## 1. Build a dummy encoder
Simulates any off-the-shelf vision backbone that outputs a flat feature vector.

In [ ]:
# Mimics ResNet-50 without its classification head
encoder = nn.Sequential(
    nn.AdaptiveAvgPool2d((1, 1)),
    nn.Flatten(),
    nn.Linear(3, 2048),
    nn.ReLU(),
)
encoder.eval()

batch  = torch.randn(4, 3, 224, 224)
feats  = encoder(batch)
print(f'encoder output shape: {feats.shape}')  # (4, 2048)

## 2. Text2ConceptAligner — forward & backward

In [ ]:
aligner = Text2ConceptAligner(source_dim = 2048, clip_dim = 512)

# forward
aligned = aligner(feats)
print(f'aligned shape : {aligned.shape}')           # (4, 512)
print(f'L2 norm       : {aligned.norm(dim=-1)}')    # should all be 1.0

# backward
target = torch.randn(4, 512)
target = torch.nn.functional.normalize(target, dim=-1)
loss   = (1 - (aligned * target).sum(dim=-1)).mean()
loss.backward()
print(f'cosine loss   : {loss.item():.4f}')         # ~1.0 for random init
print('backward ok')

## 3. Checkpoint save / load round-trip

In [ ]:
import tempfile, os

with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    ckpt_path = f.name

torch.save({
    'aligner'    : aligner.state_dict(),
    'source_dim' : aligner.source_dim,
    'clip_dim'   : aligner.clip_dim,
}, ckpt_path)

loaded = Text2ConceptAligner.from_checkpoint(ckpt_path)
print(f'source_dim : {loaded.source_dim}')   # 2048
print(f'clip_dim   : {loaded.clip_dim}')     # 512
os.unlink(ckpt_path)
print('checkpoint round-trip ok')

## 4. Text2Concept — concept_similarity, zero_shot_classify, get_cav
Uses real CLIP (ViT-B-16 / openai weights) — downloads ~350 MB on first run.

In [ ]:
t2c = Text2Concept(encoder, aligner)

# concept similarity — shape check
sims = t2c.concept_similarity(batch, ['red food', 'in a tree', 'indoors'])
print(f'concept_similarity shape : {sims.shape}')    # (4, 3)
print(f'values (cosine, [-1,1]) :\n{sims}')

# zero-shot classify
preds = t2c.zero_shot_classify(batch, ['cat', 'dog', 'bird', 'car'])
print(f'zero_shot_classify preds : {preds}')         # (4,)

# CAV in source space
cav = t2c.get_cav('stripes')
print(f'CAV shape                : {cav.shape}')     # (1, 2048)
print(f'CAV norm                 : {cav.norm():.4f}') # 1.0

# class-refined CAV
cav_refined = t2c.get_cav('in a tree', class_names=['bird', 'squirrel', 'monkey'])
print(f'refined CAV shape        : {cav_refined.shape}')  # (1, 2048)

## 5. Summary

In [ ]:
from torchinfo import summary
summary(aligner, input_size = (4, 2048))